# Generador de horarios con FET (`fet-cl`) — planilla del Encargado Docente (DISC)

Este notebook toma la planilla de trabajo del Encargado Docente (`datos cliente.xlsx`, periodo 2026-20 del DISC) y:

1. **Limpia y normaliza** los datos (ETL): RUT con formatos mezclados, docentes múltiples, horarios escritos como texto (`Ma F (V3), Mi E (V3)`), ligas entre secciones, capacidades de sala escondidas en los títulos de otra hoja, secciones inactivas o que no se dictarán.
2. **Construye el problema de timetabling** en el formato XML de FET (actividades, docentes, grupos de estudiantes, salas y restricciones).
3. **Ejecuta `fet-cl` varias veces** con distintas semillas aleatorias para obtener **varias versiones** de horario.
4. **Verifica cada versión con un verificador independiente** del motor (choques de docente, de grupo, de sala, capacidad, reglas de negocio) y calcula indicadores de calidad.
5. **Exporta cada versión como tablas** (Excel + CSV): la planilla del cliente con la columna `Horario` completada y una hoja plana de carga para Banner (una fila por NRC y sesión). El PDF ordenado por programa/docente/sala queda como opción (`GENERAR_PDF`).

> Funciona en Google Colab y en Jupyter local. Los datos del cliente **no viven en el repositorio** (contienen RUT y nombres): en Binder o Colab se te pedirá subir `datos cliente.xlsx`; en local basta con dejarlo junto al notebook.

## 0. Parámetros

In [ ]:
# ----------------------------- PARÁMETROS (editables) -----------------------------
N_VERSIONES     = 6        # cuántas versiones de horario generar
TIEMPO_LIMITE_S = 40       # segundos máximos de fet-cl por versión
SEMILLA_BASE    = 2026     # reproducibilidad: misma semilla base => mismas versiones
MODO_SALAS      = "cliente"  # "cliente": cada clase solo puede usar las salas que el cliente ya le asignó
                             # "libre":   cualquier sala con capacidad suficiente (si la clase usaba sala)
DIAS_HORA_ACOTADOS = True  # profesores hora: solo se les asignan los días en que hoy ya dictan (proxy de disponibilidad)
FIJAR_SABADO    = True     # las sesiones de sábado del cliente se mantienen en su día y bloque
MAX_HUECOS_SEMANA = 2      # tope duro de bloques libres intermedios por semana en cada semestre (None = sin tope)
MAX_VENTANAS_DOCENTE = 2   # tope duro de ventanas (bloques libres entre clases) por semana en cada docente (None = sin tope)
GENERAR_PDF     = False    # el entregable principal es Excel/CSV (tablas); el PDF es opcional, para revisar a ojo
PERIODO         = "2026-20"  # periodo académico que se incluye en las tablas de salida
ARCHIVO_DATOS   = "datos cliente.xlsx"

## 1. Entorno: FET (`fet-cl`) y librerías

In [ ]:
import os, subprocess, glob, sys, re, io, json, random, shutil, zipfile, unicodedata, datetime as dt
from concurrent.futures import ThreadPoolExecutor

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PROJECT_DIR = "/content" if IN_COLAB else os.path.abspath(os.getcwd())
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)

BIN_DIR = os.path.join(PROJECT_DIR, "fet-bin")
FET_CL  = os.path.join(BIN_DIR, "fet-cl-7.10.5", "bin", "fet-cl")

if not os.path.isfile(FET_CL):
    # build "command-line-only" (sin Qt GUI) del mirror del proyecto
    os.makedirs(BIN_DIR, exist_ok=True)
    tar_path = os.path.join(BIN_DIR, "fet-cl.tar.gz")
    subprocess.run(["curl", "-sL", "-o", tar_path,
        "https://github.com/nico-alvz/fet-mirror/releases/download/v7.10.5/fet-cl-7.10.5-linux-x86_64.tar.gz"], check=True)
    subprocess.run(["tar", "xzf", tar_path, "-C", BIN_DIR], check=True)
assert os.path.isfile(FET_CL), "No se encontró el binario fet-cl"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "openpyxl", "pandas", "reportlab", "matplotlib"], check=True)
print("Entorno:", "Google Colab" if IN_COLAB else "Jupyter local")
print("fet-cl :", FET_CL)
print(subprocess.run([FET_CL, "--version"], capture_output=True, text=True, env=dict(os.environ, QT_QPA_PLATFORM="offscreen")).stdout.strip()[:200])

## 2. ¿Hace falta un ETL? Sí

**ETL** = *Extract, Transform, Load*: extraer los datos de su fuente, **transformarlos** a un modelo limpio y consistente y **cargarlos** donde se usan (aquí, el archivo XML de FET). Es necesario porque la planilla es una herramienta de trabajo humana, no un conjunto de datos estructurado. Problemas reales detectados en `datos cliente.xlsx`:

| Problema en la planilla | Ejemplo | Tratamiento en el ETL |
|---|---|---|
| RUT con tipos mezclados (número/texto, con `K`) | `256927233`, `'24712683K'` | normalizar a texto sin puntos ni guion |
| Varios docentes en una celda, con porcentajes | `Docente A (33%) + Docente B (33%) + …` | separar en lista de docentes; todos deben estar libres en el bloque |
| Docentes sin RUT | `Nombre Apellido` sin RUT en la planilla | cruzar por nombre con la hoja `academicos`; si no existe, identificador provisional |
| Horario como texto libre | `Ma F (V3), Mi E (V3) Vi C`, `Lu DE (U8)`, `Sa BC (U8)` | parsear día + bloque(s) + sala; `DE` = sesión doble |
| Nombres de sala inconsistentes | `V3`, `V3-ROB-2`, `V2-Uni`, `V2-UNIV.` | tabla de alias → sala canónica |
| Capacidad de sala en el título de un bloque de otra hoja | `Ladeco Robótica (A-V2, V2-ROB-1): 32` | extraer con expresiones regulares |
| Secciones ligadas (mismo curso dictado junto para varios programas) | `Liga = CYBR` ×3 | fusionar en **una sola clase** (mismo docente, sala y bloque), sumando cupos |
| Filas que no se programan | `INACTIVE`, `No se dictará`, sin horario | excluir y **reportar** el motivo |
| Cupo vacío, `Block` inconsistente con el horario, saltos de línea en códigos | `"FAIC-00600\n"` | limpiar/convertir; el horario textual manda sobre `Block` |

Sin este paso, FET recibiría choques falsos (mismo docente con dos RUT), salas inexistentes o secciones duplicadas.

### 2.1 Extracción (Extract): leer la planilla

In [ ]:
import pandas as pd, openpyxl
pd.set_option("display.width", 220, "display.max_columns", 40, "display.max_colwidth", 60, "display.max_rows", 200)

def find_xlsx():
    ruta = os.path.join(PROJECT_DIR, ARCHIVO_DATOS)      # nombre exacto: nunca se adivina otro archivo
    return ruta if os.path.isfile(ruta) else None

def pedir_archivo(nombre=None):
    """En Binder/Jupyter local muestra un botón para subir el .xlsx (los datos NO viven en el repositorio)."""
    import ipywidgets as w
    from IPython.display import display
    up = w.FileUpload(accept=".xlsx", multiple=False, description="Subir .xlsx")
    def _guardar(cambio):
        v = up.value                                  # ipywidgets 8: tupla de dicts; ipywidgets 7: dict nombre -> dict
        items = [(f["name"], f["content"]) for f in v] if not isinstance(v, dict) else [(k, d["content"]) for k, d in v.items()]
        for nom, contenido in items:
            destino = nombre or nom
            with open(os.path.join(PROJECT_DIR, destino), "wb") as fh: fh.write(bytes(contenido))
            print("Guardado:", destino, "-> ahora vuelve a ejecutar esta celda")
    up.observe(_guardar, names="value"); display(up)

XLSX = find_xlsx()
if not XLSX and IN_COLAB:
    from google.colab import files
    print(f"Sube el archivo '{ARCHIVO_DATOS}':")
    files.upload(); XLSX = find_xlsx()
if not XLSX:
    print(f"Los datos del cliente NO están en el repositorio (contienen RUT y nombres). Sube '{ARCHIVO_DATOS}' con el botón,")
    print("o arrástralo al panel de archivos de JupyterLab (carpeta de este cuaderno), y vuelve a ejecutar esta celda.")
    pedir_archivo(ARCHIVO_DATOS)
    raise FileNotFoundError(f"Falta '{ARCHIVO_DATOS}' en {PROJECT_DIR}")

raw      = pd.read_excel(XLSX, sheet_name="asignaturas", dtype=object)
academ   = pd.read_excel(XLSX, sheet_name="academicos", dtype=object).dropna(how="all")
wb       = openpyxl.load_workbook(XLSX, data_only=True)
print("Archivo:", os.path.basename(XLSX), "| hojas:", wb.sheetnames)
print(f"asignaturas: {raw.shape[0]} filas x {raw.shape[1]} columnas | académicos: {len(academ)}")
raw.iloc[:6, [0, 1, 2, 3, 5, 6, 11, 12, 13, 15, 16, 20, 21, 22, 23, 24]]

### 2.2 Transformación (Transform): funciones de limpieza

In [ ]:
def norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode().lower()
    return re.sub(r"[^a-z0-9 ]+", " ", s).split()

def clean_rut(v):
    if v is None or (isinstance(v, float) and pd.isna(v)): return None
    s = re.sub(r"[^0-9kK]", "", str(v)).upper()
    return s or None

DIAS = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado"]
DIA_ABR = {"Lu": "Lunes", "Ma": "Martes", "Mi": "Miércoles", "Ju": "Jueves", "Vi": "Viernes", "Sa": "Sábado"}
DIA_CORTO = {v: k for k, v in DIA_ABR.items()}
ALIAS_SALA = {"V3": "V3-Rob", "V3-ROB-2": "V3-Rob", "V3-ROB": "V3-Rob", "V2-ROB-1": "V2-Rob", "V2-ROB": "V2-Rob",
              "V2-UNI": "V2-Univ", "V2-UNIV": "V2-Univ", "V2-UNIV.": "V2-Univ"}
def canon_sala(r): 
    r = r.strip(); return ALIAS_SALA.get(r.upper(), r)

BLOQUES = list("ABCDEFG")
SESION_RE = re.compile(r"(Lu|Ma|Mi|Ju|Vi|Sa)\s+([A-G]{1,3})\s*(?:\(([^)]+)\))?")
def parse_horario(h):
    """'Ma F (V3), Mi E (V3)' -> [(dia, [bloques], sala)]"""
    if h is None or (isinstance(h, float) and pd.isna(h)): return []
    return [(DIA_ABR[d], list(bl), canon_sala(r) if r else None) for d, bl, r in SESION_RE.findall(str(h))]

def merge_adjacent(sesiones):
    """Bloques consecutivos, mismo día, sala y turno (A–C | D–G) -> una sola sesión doble."""
    out = []
    for dia, bl, sala in sesiones:
        bl = sorted(bl)
        if out and out[-1][0] == dia and out[-1][2] == sala and BLOQUES.index(bl[0]) == BLOQUES.index(out[-1][1][-1]) + 1 \
                and BLOQUES.index(bl[0]) != 3:
            out[-1] = (dia, out[-1][1] + bl, sala)
        else:
            out.append((dia, bl, sala))
    return out

def extraer_salas_y_bloques(wb):
    """Capacidades (títulos tipo 'Redes (Y1-005): 20') y horas de cada bloque, desde la hoja 'laboratorios'."""
    ws = wb["laboratorios"]; cap = {}
    for row in ws.iter_rows():
        for c in row:
            if c.column in (2, 12) and isinstance(c.value, str) and "Bloque" not in c.value:
                m = re.match(r"^\s*(.+?)\s*:\s*(\d+)", c.value)
                if not m: continue
                nombre, n = m.group(1), int(m.group(2))
                for tok in re.findall(r"Y\d-\d+", nombre): cap[canon_sala(tok)] = n
                if "Ladeco Rob" in nombre: cap["V2-Rob"] = n
                if "Ladeco Univ" in nombre: cap["V2-Univ"] = n
                if nombre.startswith("V3-Rob"): cap["V3-Rob"] = n
    horas = {}
    for r in ws.iter_rows(min_row=3, max_row=12, min_col=2, max_col=4, values_only=True):
        if r[0] in BLOQUES and hasattr(r[1], "strftime"): horas[r[0]] = (r[1].strftime("%H:%M"), r[2].strftime("%H:%M"))
    return cap, horas

CAP_DECLARADA, HORAS = extraer_salas_y_bloques(wb)
print("Bloques:", {k: f"{a}-{b}" for k, (a, b) in HORAS.items()})
print("Capacidades declaradas por el cliente:", CAP_DECLARADA)

In [ ]:
def limpiar_secciones(raw, academ):
    a = raw.copy(); a.columns = [str(c) for c in a.columns]
    a = a.rename(columns={"Tipo.1": "Contrato"})
    for c in ["Code", "Asignatura"]:
        a[c] = a[c].astype("string").str.replace(r"\s+", " ", regex=True).str.strip()
    a["Asignatura"] = a["Asignatura"].fillna("(sin nombre)")
    a["Tipo"] = a["Tipo"].astype("string").str.strip()
    for c in ["Cupo", "Block", "Semestre"]: a[c] = pd.to_numeric(a[c], errors="coerce")
    a["Sesiones"] = a["Horario"].map(parse_horario)

    por_nombre = {tuple(norm(r["Nombre"])[:2]): clean_rut(r["RUT"]) for _, r in academ.iterrows()}
    IGNORAR = {"N. N."}   # marcador de docente por definir; las iniciales sueltas (una sola palabra) ya se descartan por len(n.split()) < 2
    def docentes(row):
        ruts = [clean_rut(x) for x in re.split(r"\s+[-+]\s+|\s*/\s*", str(row["RUT"]))] if pd.notna(row["RUT"]) else []
        nombres = re.split(r"\s*(?:\+|,)\s*", re.sub(r"\(\d+%\)", "", str(row["Profesor"]))) if pd.notna(row["Profesor"]) else []
        nombres = [n.strip() for n in nombres if n.strip()]
        out = []
        for i, n in enumerate(nombres):
            if len(n.split()) < 2 or n.upper() in IGNORAR: continue
            rut = ruts[i] if len(ruts) == len(nombres) else (ruts[0] if len(ruts) == 1 and len(nombres) == 1 else None)
            rut = rut or por_nombre.get(tuple(norm(n)[:2]))
            out.append((rut or "SINRUT-" + "".join(norm(n)), n))
        return out
    a["Docentes"] = a.apply(docentes, axis=1)

    a["Motivo"] = ""
    a.loc[a["Estado"].astype(str).str.upper().eq("INACTIVE"), "Motivo"] = "estado INACTIVE"
    a.loc[a["Resolución CAD"].astype(str).eq("No se dictará"), "Motivo"] = "CAD: no se dictará"
    a.loc[(a["Motivo"] == "") & (a["Sesiones"].map(len) == 0) & a["Block"].isna(), "Motivo"] = "sin horario ni bloques"
    return a

secciones = limpiar_secciones(raw, academ)
print("Secciones leídas:", len(secciones))
print(secciones["Motivo"].replace("", "→ SE PROGRAMA").value_counts().to_string())

### 2.3 Fusión de ligas y construcción de las *clases* a programar

In [ ]:
def construir_clases(sec):
    """Una *clase* = unidad que FET debe ubicar. Las secciones ligadas (mismo curso, docente y horario) se fusionan."""
    v = sec[sec["Motivo"] == ""].copy()
    v["Ses"] = v["Sesiones"].map(merge_adjacent)
    v["SesFallback"] = v.apply(lambda r: [(None, ["A"], None)] * int(r["Block"]) if not r["Ses"] and pd.notna(r["Block"]) else [], axis=1)
    def firma(r):
        return (r["Liga"] if pd.notna(r["Liga"]) else f"_{r.name}", r["P"], tuple(t[0] for t in r["Docentes"]),
                tuple((d, tuple(b)) for d, b, _ in r["Ses"]))
    v["_k"] = v.apply(firma, axis=1)
    filas = []
    for k, g in v.groupby("_k", sort=False):
        r0 = g.iloc[0]
        grupos = sorted({(p, int(sm)) for p, sm, t in zip(g["Programa"], g["Semestre"], g["Tipo"])
                         if pd.notna(sm) and pd.notna(p) and "/" not in str(p)
                         and t in ("Obligatorio", "Taller", "Proyecto", "Cátedra", "Laboratorio")})
        filas.append(dict(
            ClaseId=len(filas) + 1, Filas=list(g.index), NRC=" / ".join(map(str, g["NRC"])), Codigo=" / ".join(g["Code"].dropna().unique()),
            Asignatura=r0["Asignatura"], Seccion=r0["P"], Tipo=r0["Tipo"],
            Programas=" / ".join(sorted(set(g["Programa"].dropna()))), Cupo=int(g["Cupo"].fillna(0).sum()),
            Docentes=r0["Docentes"], Contrato=r0["Contrato"] if pd.notna(r0["Contrato"]) else "",
            Grupos=grupos, SesionesCliente=r0["Ses"] or r0["SesFallback"],
            SalasCliente=sorted({sa for ses in g["Ses"] for _, _, sa in ses if sa})))
    return pd.DataFrame(filas)

clases = construir_clases(secciones)
print(f"{(secciones.Motivo == '').sum()} secciones válidas -> {len(clases)} clases "
      f"({sum(len(s) for s in clases.SesionesCliente)} sesiones semanales a ubicar)")
clases.assign(Docentes=clases.Docentes.map(lambda l: ", ".join(n for _, n in l)),
              Sesiones=clases.SesionesCliente.map(lambda l: " ".join(f"{DIA_CORTO.get(d, '?')} {''.join(b)}" for d, b, _ in l)),
              Salas=clases.SalasCliente.map(", ".join))[["ClaseId", "NRC", "Asignatura", "Seccion", "Cupo", "Docentes", "Sesiones", "Salas"]].head(25)

### 2.4 Carga (Load) — salas, docentes y reporte de calidad del ETL

In [ ]:
# ---- salas: capacidad efectiva = max(declarada, mayor cupo que el propio cliente ya ubicó allí)
uso = {}
for _, c in clases.iterrows():
    for sa in c.SalasCliente: uso[sa] = max(uso.get(sa, 0), c.Cupo)
SALAS = sorted(set(uso) | set(CAP_DECLARADA))
# La capacidad DECLARADA por el cliente es siempre el límite. Solo si una sala no trae dato se ESTIMA con el mayor
# cupo que el cliente ya ubicó allí (y se marca como estimada, pendiente de confirmar).
CAP_EFECTIVA = {s: CAP_DECLARADA[s] if s in CAP_DECLARADA else max(uso.get(s, 0), 10) for s in SALAS}
salas_df = pd.DataFrame({"Sala": SALAS,
    "Capacidad declarada": [CAP_DECLARADA.get(s) for s in SALAS],
    "Mayor cupo ubicado por el cliente": [uso.get(s) for s in SALAS],
    "Capacidad usada como límite": [CAP_EFECTIVA[s] for s in SALAS]})
salas_df["Origen"] = ["declarada" if s in CAP_DECLARADA else "ESTIMADA (confirmar con el cliente)" for s in SALAS]
salas_df["Alerta"] = [f"el cliente ya ubica cupo {uso[s]} > capacidad declarada {CAP_DECLARADA[s]}" if s in CAP_DECLARADA and uso.get(s, 0) > CAP_DECLARADA[s] else "" for s in SALAS]
print("Salas (la capacidad declarada es el límite; las salas sin dato llevan una capacidad ESTIMADA):")
display(salas_df)

# ---- docentes
DOCENTES = {}
for _, c in clases.iterrows():
    for rut, nombre in c.Docentes:
        d = DOCENTES.setdefault(rut, dict(rut=rut, nombre=nombre, contrato=set(), dias=set()))
        if c.Contrato: d["contrato"].add(c.Contrato)
        d["dias"] |= {dia for dia, _, _ in c.SesionesCliente if dia}
for d in DOCENTES.values(): d["contrato"] = "/".join(sorted(d["contrato"])) or "s/i"

print(f"{len(DOCENTES)} docentes en el modelo | sin RUT en la planilla ni en 'academicos': "
      f"{[d['nombre'] for d in DOCENTES.values() if d['rut'].startswith('SINRUT')]}")
def rut_valido(r):
    cuerpo, dv = r[:-1], r[-1]
    if not cuerpo.isdigit(): return False
    suma, m = 0, 2
    for d in reversed(cuerpo): suma += int(d) * m; m = 2 if m == 7 else m + 1
    esp = 11 - suma % 11
    return dv == ("0" if esp == 11 else "K" if esp == 10 else str(esp))
rut_malos = [(d["nombre"], d["rut"]) for d in DOCENTES.values() if not d["rut"].startswith("SINRUT") and not rut_valido(d["rut"])]
print("RUT con dígito verificador inválido (revisar en la planilla):", rut_malos or "ninguno")
excl = secciones[secciones.Motivo != ""][["NRC", "Code", "Asignatura", "Motivo"]]
print(f"\nSecciones EXCLUIDAS de la programación ({len(excl)}) — el ETL las reporta para que el cliente las revise:")
display(excl.reset_index(drop=True))

## 3. Modelo de FET (archivo `.fet`)

Cada **sesión semanal** de una clase es una *actividad* de FET (una sesión doble `DE` = actividad de duración 2). Reglas incorporadas:

**Duras (deben cumplirse siempre)**
- Un docente no puede estar en dos actividades a la vez (con varios docentes en una clase, todos deben estar libres).
- Un grupo de estudiantes (*programa + semestre*) no puede tener dos asignaturas a la vez → **sin choques dentro de un semestre**. Las secciones paralelas (`C2`, `T1…T5`, laboratorios `L01…`) son subgrupos del semestre.
- Una sala no puede alojar dos clases a la vez, y solo se usan salas con capacidad ≥ cupo (ver modo de salas).
- Sesiones de una misma clase en días distintos; las sesiones dobles no cruzan el almuerzo (entre los bloques C y D).
- Solo lunes a viernes; las sesiones de sábado del cliente se mantienen fijas (`FIJAR_SABADO`).
- Profesores hora: solo los días en que hoy ya dictan (`DIAS_HORA_ACOTADOS`), como aproximación de su disponibilidad declarada.

**Compacidad (opcional, duras si se activan)**: tope de bloques libres por semana en cada semestre (`MAX_HUECOS_SEMANA`) y de ventanas por semana en cada docente (`MAX_VENTANAS_DOCENTE`).

**Indicadores** (los mide el verificador): bloques libres de estudiantes, ventanas de docentes, cercanía al horario original.

In [ ]:
from xml.sax.saxutils import escape as x

DIA_IDX_ = {d: i for i, d in enumerate(DIAS)}
def hora_nombre(i): 
    b = BLOQUES[i]; return f"{b} {HORAS[b][0]}-{HORAS[b][1]}"
HORAS_FET = [hora_nombre(i) for i in range(len(BLOQUES))]
N_H = len(BLOQUES)

def inicios_validos(dur):
    """Bloques de inicio permitidos: la sesión cabe en el día y no cruza el almuerzo (entre C y D)."""
    return [h for h in range(N_H - dur + 1) if not (h < 3 < h + dur)]

def nombre_docente(rut): return DOCENTES[rut]["nombre"]
def nombre_year(p, s): return f"{p}-S{s}"
def sets_estudiantes(c):
    if not c.Grupos: return []
    return [nombre_year(p, s) if c.Seccion == "C1" else f"{nombre_year(p, s)}-{c.Seccion}" for p, s in c.Grupos]

def salas_candidatas(c):
    if not c.SalasCliente: return []          # p. ej. Práctica Pre-Profesional: sin sala
    if MODO_SALAS == "cliente":
        cand = [s for s in c.SalasCliente if CAP_EFECTIVA[s] >= c.Cupo]
        if cand: return cand
        # las salas del cliente no alcanzan (p. ej. cupo 21 en una sala declarada de 20): se buscan otras con capacidad
    return [s for s in SALAS if CAP_EFECTIVA[s] >= c.Cupo]     # [] = ninguna sala alcanza: se reporta como choque

def construir_actividades():
    acts, gid = [], 0
    for _, c in clases.iterrows():
        n = len(c.SesionesCliente); ids = []
        for k, (dia, bl, sala) in enumerate(c.SesionesCliente):
            gid += 1; ids.append(gid)
            acts.append(dict(Id=gid, ClaseId=c.ClaseId, Sesion=k + 1, Dur=len(bl), DiaCliente=dia,
                             BloqueCliente=BLOQUES.index(bl[0]), SalaCliente=sala,
                             Fijo=bool(FIJAR_SABADO and dia == "Sábado"),
                             GrupoId=0 if n == 1 else ids[0], TotalDur=sum(len(s[1]) for s in c.SesionesCliente)))
    return pd.DataFrame(acts)

acts = construir_actividades()
clase_por_id = {c.ClaseId: c for _, c in clases.iterrows()}
print(len(acts), "actividades")

In [ ]:
def escribir_fet(path, fijos=None, permitidos=None, salas_forzadas=None):
    """fijos: {IdActividad: (dia_idx, hora_idx, sala)} bloqueadas; permitidos: {Id: set((dia_idx, hora))} inicios permitidos;
    salas_forzadas: {Id: [salas]}. Sirven para la replanificación parcial (sección 7)."""
    fijos, permitidos, salas_forzadas = fijos or {}, permitidos or {}, salas_forzadas or {}
    P = []; w = P.append
    w('<?xml version="1.0" encoding="UTF-8"?>\n<fet version="7.10.5">\n<Mode>Official</Mode>\n')
    w("<Institution_Name>UCN - DISC - Oferta 2026-20</Institution_Name>\n<Comments>Generado desde datos cliente.xlsx</Comments>\n")
    w(f"<Days_List>\n<Number_of_Days>{len(DIAS)}</Number_of_Days>\n" + "".join(f"<Day>\n\t<Name>{d}</Name>\n</Day>\n" for d in DIAS) + "</Days_List>\n")
    w(f"<Hours_List>\n<Number_of_Hours>{N_H}</Number_of_Hours>\n" + "".join(f"<Hour>\n\t<Name>{h}</Name>\n</Hour>\n" for h in HORAS_FET) + "</Hours_List>\n")
    asigs = clases.drop_duplicates("Asignatura")
    w("<Subjects_List>\n" + "".join(f"<Subject>\n\t<Name>{x(a)}</Name>\n\t<Comments></Comments>\n</Subject>\n" for a in asigs.Asignatura) + "</Subjects_List>\n")
    w("<Activity_Tags_List>\n</Activity_Tags_List>\n<Buildings_List>\n</Buildings_List>\n")
    w("<Teachers_List>\n")
    for rut, d in DOCENTES.items():
        w(f"<Teacher>\n\t<Name>{x(d['nombre'])} [{rut}]</Name>\n\t<Target_Number_of_Hours>0</Target_Number_of_Hours>\n"
          f"\t<Qualified_Subjects>\n\t</Qualified_Subjects>\n\t<Comments>{x(d['contrato'])}</Comments>\n</Teacher>\n")
    w("</Teachers_List>\n<Rooms_List>\n")
    for s in SALAS:
        w(f"<Room>\n\t<Name>{x(s)}</Name>\n\t<Building></Building>\n\t<Capacity>{CAP_EFECTIVA[s]}</Capacity>\n\t<Virtual>false</Virtual>\n\t<Comments></Comments>\n</Room>\n")
    w("</Rooms_List>\n")
    # estudiantes: un Year por (programa, semestre); las secciones paralelas son Groups del Year.
    years = {}
    for _, c in clases.iterrows():
        for p, s in c.Grupos:
            yr = years.setdefault(nombre_year(p, s), set())
            if c.Seccion != "C1": yr.add(f"{nombre_year(p, s)}-{c.Seccion}")
    w("<Students_List>\n")
    for yn in sorted(years):
        w(f"<Year>\n\t<Name>{yn}</Name>\n\t<Number_of_Students>0</Number_of_Students>\n\t<Comments></Comments>\n\t<Number_of_Categories>0</Number_of_Categories>\n\t<Separator> </Separator>\n")
        for g in sorted(years[yn]):
            w(f"\t<Group>\n\t\t<Name>{g}</Name>\n\t\t<Number_of_Students>0</Number_of_Students>\n\t\t<Comments></Comments>\n\t</Group>\n")
        w("</Year>\n")
    w("</Students_List>\n<Activities_List>\n")
    for a in acts.itertuples():
        c = clase_por_id[a.ClaseId]
        w("<Activity>\n" + "".join(f"\t<Teacher>{x(nombre_docente(r))} [{r}]</Teacher>\n" for r, _ in c.Docentes)
          + f"\t<Subject>{x(c.Asignatura)}</Subject>\n" + "".join(f"\t<Students>{e}</Students>\n" for e in sets_estudiantes(c))
          + f"\t<Duration>{a.Dur}</Duration>\n\t<Total_Duration>{a.TotalDur}</Total_Duration>\n\t<Id>{a.Id}</Id>\n"
          + f"\t<Activity_Group_Id>{a.GrupoId}</Activity_Group_Id>\n\t<Active>true</Active>\n\t<Comments>{x(c.Codigo)}</Comments>\n</Activity>\n")
    w("</Activities_List>\n")

    # ---------------- restricciones de tiempo
    w("<Time_Constraints_List>\n<ConstraintBasicCompulsoryTime>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintBasicCompulsoryTime>\n")
    for a in acts.itertuples():
        if a.Id in fijos:
            d_, h_, _ = fijos[a.Id]
            w(f"<ConstraintActivityPreferredStartingTime>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Activity_Id>{a.Id}</Activity_Id>\n"
              f"\t<Day>{DIAS[d_]}</Day>\n\t<Hour>{HORAS_FET[h_]}</Hour>\n\t<Permanently_Locked>false</Permanently_Locked>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintActivityPreferredStartingTime>\n")
        elif a.Fijo:
            w(f"<ConstraintActivityPreferredStartingTime>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Activity_Id>{a.Id}</Activity_Id>\n"
              f"\t<Day>{a.DiaCliente}</Day>\n\t<Hour>{HORAS_FET[a.BloqueCliente]}</Hour>\n\t<Permanently_Locked>false</Permanently_Locked>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintActivityPreferredStartingTime>\n")
        else:
            st = [(d, h) for d in DIAS[:5] for h in inicios_validos(a.Dur)]
            if a.Id in permitidos:
                st = [(d, h) for d, h in st if (DIA_IDX_[d], h) in permitidos[a.Id]]
                if not st: raise ValueError(f"Los cambios dejan sin ningún horario posible a la actividad {a.Id} ({clase_por_id[a.ClaseId].Asignatura})")
            w(f"<ConstraintActivityPreferredStartingTimes>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Activity_Id>{a.Id}</Activity_Id>\n\t<Number_of_Preferred_Starting_Times>{len(st)}</Number_of_Preferred_Starting_Times>\n"
              + "".join(f"\t<Preferred_Starting_Time>\n\t\t<Day>{d}</Day>\n\t\t<Hour>{HORAS_FET[h]}</Hour>\n\t</Preferred_Starting_Time>\n" for d, h in st)
              + "\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintActivityPreferredStartingTimes>\n")
    for cid, g in acts.groupby("ClaseId"):
        if len(g) > 1:
            w(f"<ConstraintMinDaysBetweenActivities>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Consecutive_If_Same_Day>true</Consecutive_If_Same_Day>\n\t<Number_of_Activities>{len(g)}</Number_of_Activities>\n"
              + "".join(f"\t<Activity_Id>{i}</Activity_Id>\n" for i in g.Id) + "\t<MinDays>1</MinDays>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintMinDaysBetweenActivities>\n")
    if DIAS_HORA_ACOTADOS:
        for rut, d in DOCENTES.items():
            if "Hora" in d["contrato"] and d["dias"]:
                libres = [dd for dd in DIAS if dd not in d["dias"]]
                if libres:
                    w(f"<ConstraintTeacherNotAvailableTimes>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Teacher>{x(d['nombre'])} [{rut}]</Teacher>\n"
                      f"\t<Number_of_Not_Available_Times>{len(libres) * N_H}</Number_of_Not_Available_Times>\n"
                      + "".join(f"\t<Not_Available_Time>\n\t\t<Day>{dd}</Day>\n\t\t<Hour>{h}</Hour>\n\t</Not_Available_Time>\n" for dd in libres for h in HORAS_FET)
                      + "\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintTeacherNotAvailableTimes>\n")
    if MAX_HUECOS_SEMANA is not None:
        for yn in sorted(years):
            w(f"<ConstraintStudentsSetMaxGapsPerWeek>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Max_Gaps>{MAX_HUECOS_SEMANA}</Max_Gaps>\n\t<Students>{yn}</Students>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintStudentsSetMaxGapsPerWeek>\n")
    if MAX_VENTANAS_DOCENTE is not None:
        w(f"<ConstraintTeachersMaxGapsPerWeek>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Max_Gaps>{MAX_VENTANAS_DOCENTE}</Max_Gaps>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintTeachersMaxGapsPerWeek>\n")
    w("</Time_Constraints_List>\n")

    # ---------------- restricciones de espacio
    w("<Space_Constraints_List>\n<ConstraintBasicCompulsorySpace>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintBasicCompulsorySpace>\n")
    for a in acts.itertuples():
        if a.Id in fijos:
            cand = [fijos[a.Id][2]] if fijos[a.Id][2] else []
        else:
            cand = salas_forzadas.get(a.Id) or salas_candidatas(clase_por_id[a.ClaseId])
        if cand:
            w(f"<ConstraintActivityPreferredRooms>\n\t<Weight_Percentage>100</Weight_Percentage>\n\t<Activity_Id>{a.Id}</Activity_Id>\n\t<Number_of_Preferred_Rooms>{len(cand)}</Number_of_Preferred_Rooms>\n"
              + "".join(f"\t<Preferred_Room>{x(s)}</Preferred_Room>\n" for s in cand) + "\t<Active>true</Active>\n\t<Comments></Comments>\n</ConstraintActivityPreferredRooms>\n")
    w("</Space_Constraints_List>\n<Timetable_Generation_Options_List>\n</Timetable_Generation_Options_List>\n</fet>\n")
    with open(path, "w", encoding="utf-8") as f: f.write("".join(P))

RUN_DIR = os.path.join(PROJECT_DIR, "fet_run_aldo")
os.makedirs(RUN_DIR, exist_ok=True)
FET_INPUT = os.path.join(RUN_DIR, "oferta_2026-20.fet")
escribir_fet(FET_INPUT)
print("Archivo FET escrito:", FET_INPUT, f"({os.path.getsize(FET_INPUT)/1024:.0f} KB)")

## 4. Generar varias versiones con `fet-cl`

FET es un algoritmo heurístico con componente aleatoria: **la misma entrada con distintas semillas produce horarios distintos y todos válidos**. Se ejecuta `fet-cl` una vez por versión, cada una con sus 6 componentes de semilla (derivadas de `SEMILLA_BASE`, por lo que el resultado es reproducible), en paralelo.

In [ ]:
def semillas(i):
    r = random.Random(SEMILLA_BASE * 1000 + i)
    return [r.randint(1, 4294967000) for _ in range(3)] + [r.randint(1, 4294944000) for _ in range(3)]

def ejecutar_version(i, fet_input=None, tag="v", offset=0):
    fet_input = fet_input or FET_INPUT
    out = os.path.join(RUN_DIR, f"{tag}{i:02d}"); shutil.rmtree(out, ignore_errors=True); os.makedirs(out)
    s = semillas(i + offset)
    cmd = [FET_CL, f"--inputfile={fet_input}", f"--outputdir={out}", f"--timelimitseconds={TIEMPO_LIMITE_S}",
           "--writetimetablesxml=true", "--writetimetablesactivities=true", "--writetimetableconflicts=true",
           "--writetimetablesstatistics=false", "--writetimetablesdayshorizontal=false", "--writetimetablesdaysvertical=false",
           "--writetimetablestimehorizontal=false", "--writetimetablestimevertical=false", "--writetimetablessubgroups=false",
           "--writetimetablesgroups=false", "--writetimetablesyears=false", "--writetimetablesteachers=false",
           "--writetimetablesteachersfreeperiods=false", "--writetimetablesbuildings=false", "--writetimetablesrooms=false",
           "--writetimetablessubjects=false", "--writetimetablesactivitytags=false", "--htmllevel=0",
           "--randomseeds10=%d" % s[0], "--randomseeds11=%d" % s[1], "--randomseeds12=%d" % s[2],
           "--randomseeds20=%d" % s[3], "--randomseeds21=%d" % s[4], "--randomseeds22=%d" % s[5]]
    t0 = dt.datetime.now()
    r = subprocess.run(cmd, capture_output=True, text=True, env=dict(os.environ, QT_QPA_PLATFORM="offscreen"))
    ok = "Generation successful" in r.stdout
    xmls = glob.glob(os.path.join(out, "timetables", "*", "*_activities.xml"))
    return dict(version=i, tag=tag, ok=ok and bool(xmls), seg=(dt.datetime.now() - t0).total_seconds(), xml=xmls[0] if xmls else None, semillas=s,
                log=r.stdout[-400:] + r.stderr[-200:])

print(f"Ejecutando {N_VERSIONES} versiones (límite {TIEMPO_LIMITE_S}s c/u) …")
with ThreadPoolExecutor(max_workers=min(N_VERSIONES, os.cpu_count() or 2)) as ex:
    corridas = list(ex.map(ejecutar_version, range(1, N_VERSIONES + 1)))
for c in corridas:
    print(f"  v{c['version']:02d}: {'OK ' if c['ok'] else 'SIN SOLUCIÓN'}  {c['seg']:.1f}s  semillas={c['semillas']}")
    if not c["ok"]: print("      ", c["log"].replace("\n", " ")[-300:])
assert any(c["ok"] for c in corridas), "Ninguna versión encontró solución: relaja restricciones (p. ej. MODO_SALAS='libre') o sube TIEMPO_LIMITE_S"

## 5. Leer las soluciones y **verificarlas de forma independiente**

El verificador no usa nada de FET: recalcula desde la solución (día, bloque, sala) todos los **choques** (docente, estudiantes, sala) y las demás reglas duras (capacidad, completitud, topes). Una versión con choques **no se considera aprobable ni se exporta** (HOR-003 y HOR-006 del ERS). «Aprobable» significa *factible bajo los supuestos declarados*, no *validada para operación*. También calcula indicadores de calidad para comparar versiones.

In [ ]:
import xml.etree.ElementTree as ET
DIA_IDX = {d: i for i, d in enumerate(DIAS)}
HORA_IDX = {h: i for i, h in enumerate(HORAS_FET)}

def cargar_solucion(xml_path):
    sol = {}
    for a in ET.parse(xml_path).getroot().findall("Activity"):
        sol[int(a.find("Id").text)] = (DIA_IDX[a.find("Day").text], HORA_IDX[a.find("Hour").text],
                                       (a.find("Room").text or "") if a.find("Room") is not None else "")
    return sol

def tabla_horario(sol):
    """DataFrame con una fila por sesión ubicada."""
    filas = []
    for a in acts.itertuples():
        c = clase_por_id[a.ClaseId]; d, h, sala = sol[a.Id]
        filas.append(dict(Id=a.Id, ClaseId=a.ClaseId, NRC=c.NRC, Codigo=c.Codigo, Asignatura=c.Asignatura, Seccion=c.Seccion,
                          Programas=c.Programas, Cupo=c.Cupo, Docentes=", ".join(n for _, n in c.Docentes),
                          RUTs=[r for r, _ in c.Docentes], Sets=sets_estudiantes(c), Dia=d, Hora=h, Dur=a.Dur, Sala=sala,
                          DiaCliente=a.DiaCliente, HoraCliente=a.BloqueCliente, Sesion=a.Sesion))
    return pd.DataFrame(filas)

def verificar(t, reglas_de_modelo=True):
    """Devuelve (lista de choques y reglas incumplidas, dict de indicadores)."""
    viol = []
    # completitud: cada actividad del problema aparece exactamente una vez, con su duración
    esperado = acts.set_index("Id")["Dur"].to_dict()
    if sorted(t.Id) != sorted(esperado): viol.append(f"Solución incompleta o duplicada: faltan {sorted(set(esperado) - set(t.Id))[:5]}, sobran {sorted(set(t.Id) - set(esperado))[:5]}")
    if any(esperado.get(r.Id) != r.Dur for r in t.itertuples()): viol.append("Duración de una sesión distinta de la esperada")
    ocup_doc, ocup_sala, ocup_est = {}, {}, {}
    for r in t.itertuples():
        for hh in range(r.Hora, r.Hora + r.Dur):
            for rut in r.RUTs:
                ocup_doc.setdefault((rut, r.Dia, hh), []).append(r)
            if r.Sala: ocup_sala.setdefault((r.Sala, r.Dia, hh), []).append(r)
            for st in r.Sets:
                yr = st.split("-")[0] + "-" + st.split("-")[1]
                ocup_est.setdefault((yr, r.Dia, hh), []).append((st, r))
        # reglas por sesión
        if reglas_de_modelo and r.Dia > 4 and not (FIJAR_SABADO and r.DiaCliente == "Sábado"): viol.append(f"Sesión en sábado no permitida: {r.Asignatura} ({r.NRC})")
        if r.Hora <= 2 and r.Hora + r.Dur - 1 >= 3: viol.append(f"Sesión cruza el almuerzo (entre bloques C y D): {r.Asignatura}")
        if r.Hora + r.Dur > N_H: viol.append(f"Sesión se sale del día: {r.Asignatura}")
        if clase_por_id[r.ClaseId].SalasCliente and not r.Sala: viol.append(f"Clase sin sala: {r.Asignatura} ({r.NRC})")
        if r.Sala and r.Cupo > CAP_EFECTIVA[r.Sala]: viol.append(f"Sobrecupo de sala: {r.Asignatura} (cupo {r.Cupo}) en {r.Sala} (capacidad {CAP_EFECTIVA[r.Sala]})")
        if reglas_de_modelo and r.Sala and r.Sala not in salas_candidatas(clase_por_id[r.ClaseId]): viol.append(f"Sala no permitida por MODO_SALAS: {r.Asignatura} en {r.Sala}")
        if reglas_de_modelo and DIAS_HORA_ACOTADOS:
            for rut in r.RUTs:
                d = DOCENTES[rut]
                if "Hora" in d["contrato"] and d["dias"] and DIAS[r.Dia] not in d["dias"]:
                    viol.append(f"Profesor hora fuera de sus días: {d['nombre']} el {DIAS[r.Dia]}")
        if reglas_de_modelo and FIJAR_SABADO and r.DiaCliente == "Sábado" and (r.Dia, r.Hora) != (5, r.HoraCliente): viol.append(f"Sábado fijo movido: {r.Asignatura}")
    for (rut, d, h), l in ocup_doc.items():
        if len(l) > 1: viol.append(f"Choque de docente {DOCENTES[rut]['nombre']} ({DIAS[d]} {BLOQUES[h]}): " + " vs ".join(sorted({f'{q.Asignatura} {q.Seccion}' for q in l})))
    for (s, d, h), l in ocup_sala.items():
        if len(l) > 1: viol.append(f"Choque de sala {s} ({DIAS[d]} {BLOQUES[h]}): " + " vs ".join(sorted({q.Asignatura for q in l})))
    for (yr, d, h), l in ocup_est.items():
        # dentro de un semestre: choca si hay 2+ actividades y alguna es del nivel del semestre completo,
        # o si dos comparten el mismo subgrupo
        if len(l) > 1:
            nombres = [s for s, _ in l]; ids = {q.Id for _, q in l}
            if len(ids) > 1 and (yr in nombres or len(set(nombres)) < len(nombres)):
                viol.append(f"Choque de estudiantes {yr} ({DIAS[d]} {BLOQUES[h]}): " + " vs ".join(sorted({q.Asignatura for _, q in l})))
    # sesiones de una misma clase en días distintos
    for cid, g in t.groupby("ClaseId"):
        if len(g) > 1 and g.Dia.duplicated().any(): viol.append(f"Dos sesiones el mismo día: {g.iloc[0].Asignatura} ({g.iloc[0].NRC})")

    # ---------------- indicadores de calidad
    huecos = 0
    por_year = {}
    for (yr, d, h), l in ocup_est.items():
        if any(st == yr for st, _ in l): por_year.setdefault((yr, d), []).append(h)
    for hs in por_year.values():
        hs = sorted(hs); huecos += (hs[-1] - hs[0] + 1) - len(hs)
    por_doc = {}
    for (rut, d, h) in ocup_doc: por_doc.setdefault((rut, d), []).append(h)
    ventanas = sum((max(hs) - min(hs) + 1) - len(hs) for hs in por_doc.values())
    dias_por_doc = {}
    for (rut, d) in por_doc: dias_por_doc[rut] = dias_por_doc.get(rut, 0) + 1
    dias_doc = [dias_por_doc.get(r, 0) for r in DOCENTES]
    if reglas_de_modelo and MAX_HUECOS_SEMANA is not None:
        por_y = {}
        for (yr, d), hs in por_year.items(): por_y[yr] = por_y.get(yr, 0) + (max(hs) - min(hs) + 1) - len(set(hs))
        for yr, n in por_y.items():
            if n > MAX_HUECOS_SEMANA: viol.append(f"Tope de bloques libres superado en {yr}: {n} > {MAX_HUECOS_SEMANA}")
    if reglas_de_modelo and MAX_VENTANAS_DOCENTE is not None:
        por_d = {}
        for (rut, d), hs in por_doc.items(): por_d[rut] = por_d.get(rut, 0) + (max(hs) - min(hs) + 1) - len(set(hs))
        for rut, n in por_d.items():
            if n > MAX_VENTANAS_DOCENTE: viol.append(f"Tope de ventanas superado para {DOCENTES[rut]['nombre']}: {n} > {MAX_VENTANAS_DOCENTE}")
    con_cliente = t[t.DiaCliente.notna()]
    igual = ((con_cliente.Dia.map(lambda i: DIAS[i]) == con_cliente.DiaCliente) & (con_cliente.Hora == con_cliente.HoraCliente)).mean()
    viol = list(dict.fromkeys(viol))
    ind = {"Choques": len(viol), "Bloques libres estudiantes": huecos, "Ventanas docentes": ventanas,
           "Días promedio por docente": round(sum(dias_doc) / len(dias_doc), 2),
           "Sesiones en bloques F-G": int((t.Hora >= 5).sum()), "% igual al horario del cliente": round(100 * igual, 1)}
    return viol, ind

VERSIONES = {}
for c in corridas:
    if not c["ok"]: continue
    t = tabla_horario(cargar_solucion(c["xml"]))
    viol, ind = verificar(t)
    VERSIONES[c["version"]] = dict(t=t, viol=viol, ind=ind, corrida=c)

# ---- referencia: el horario que hoy trae la planilla del cliente, pasado por el mismo verificador
sol_cliente = {a.Id: (DIA_IDX[a.DiaCliente], a.BloqueCliente, a.SalaCliente if isinstance(a.SalaCliente, str) else "")
               for a in acts.itertuples() if isinstance(a.DiaCliente, str)}
if len(sol_cliente) == len(acts):
    t_cli = tabla_horario(sol_cliente); v_cli, i_cli = verificar(t_cli, reglas_de_modelo=False)
    print(f"Horario original del cliente: {i_cli['Choques']} choques según el verificador (posibles: algunos podrían ser clases conjuntas sin liga declarada), p. ej.:")
    for m in v_cli[:8]: print("   -", m)

resumen = pd.DataFrame({f"V{v:02d}": d["ind"] for v, d in VERSIONES.items()}).T
if len(sol_cliente) == len(acts): resumen.loc["Cliente (ref.)"] = i_cli
resumen["Puntaje (menor = mejor)"] = resumen["Choques"] * 1000 + resumen["Bloques libres estudiantes"] + 0.5 * resumen["Ventanas docentes"]
resumen["Ranking"] = resumen.drop(index="Cliente (ref.)", errors="ignore")["Puntaje (menor = mejor)"].rank(method="min")
resumen["Ranking"] = resumen["Ranking"].astype("Int64")
print("Resumen de versiones (verificador independiente de FET):")
display(resumen.sort_values("Ranking", na_position="last"))
for v, d in VERSIONES.items():
    if d["viol"]: print(f"V{v:02d}: {len(d['viol'])} choques →", d["viol"][:5])
print("Versiones distintas entre sí:", len({tuple(sorted((r.Id, r.Dia, r.Hora, r.Sala) for r in d["t"].itertuples())) for d in VERSIONES.values()}), "de", len(VERSIONES))

### Vista rápida de una versión (la mejor del ranking)

In [ ]:
import matplotlib.pyplot as plt
mejor = int(resumen["Ranking"].astype(float).idxmin()[1:])

def grafico_semanal(t, columna, valor, titulo):
    sub = t[t[columna].map(lambda z: valor in z if isinstance(z, list) else z == valor)]
    fig, ax = plt.subplots(figsize=(11, 4.8))
    ax.set_xlim(0, 5); ax.set_ylim(0, N_H); ax.invert_yaxis()
    ax.set_xticks([i + .5 for i in range(5)]); ax.set_xticklabels(DIAS[:5])
    ax.set_yticks([i + .5 for i in range(N_H)]); ax.set_yticklabels(HORAS_FET, fontsize=8); ax.grid(True, color="#ddd")
    for r in sub.itertuples():
        if r.Dia > 4: continue
        ax.add_patch(plt.Rectangle((r.Dia + .03, r.Hora + .04), .94, r.Dur - .08, color=plt.cm.tab20(r.ClaseId % 20), alpha=.9))
        ax.text(r.Dia + .5, r.Hora + r.Dur / 2, f"{r.Asignatura[:26]}\n{r.Sala or ''}", ha="center", va="center", fontsize=7)
    ax.set_title(titulo); plt.tight_layout(); plt.show()

tm = VERSIONES[mejor]["t"]
grafico_semanal(tm, "Sets", "ICCI-S6", f"V{mejor:02d} — ICCI, semestre 6 (obligatorias)")

### Pruebas del propio verificador (¿detecta de verdad los errores?)

Se alteran a propósito copias en memoria de una versión aprobable; el verificador **debe** reportar choques en cada caso. Si alguna prueba no los detecta, el notebook se detiene.

In [ ]:
def pruebas_verificador():
    base = next((d["t"] for d in VERSIONES.values() if not d["viol"]), None)
    if base is None: print("Sin versión aprobable para probar el verificador"); return
    casos = {}
    t1 = base.copy(); t1["Sala"] = ""; casos["vaciar todas las salas"] = t1
    casos["eliminar una actividad"] = base.iloc[1:].copy()
    t3 = base.copy(); fila = t3[t3.RUTs.map(len) > 0]
    a_, b_ = fila.iloc[0], fila[fila.RUTs.map(lambda L: L[0]) == fila.iloc[0].RUTs[0]].iloc[-1]
    if a_.Id != b_.Id: t3.loc[t3.Id == b_.Id, ["Dia", "Hora"]] = [a_.Dia, a_.Hora]; casos["mismo docente en dos clases a la vez"] = t3
    t4 = base.copy(); i = t4[(t4.Sala != "")].index[0]; t4.loc[i, "Sala"] = min(CAP_EFECTIVA, key=CAP_EFECTIVA.get); t4.loc[i, "Cupo"] = 99; casos["sobrecupo de sala"] = t4
    t5 = base.copy(); i = t5[t5.Dur == 1].index[0]; t5.loc[i, "Hora"] = 2; t5.loc[t5.index[t5.Dur == 2][0], "Hora"] = 2; casos["sesión doble cruzando el almuerzo"] = t5
    for nombre, tt in casos.items():
        v_, _ = verificar(tt)
        print(f"  {'OK ' if v_ else 'FALLA'} detecta «{nombre}»: {len(v_)} choque(s)")
        assert v_, f"El verificador no detectó: {nombre}"
pruebas_verificador()

## 6. Exportar tablas para el cliente y para Banner (Excel + CSV)

Según el ERS (HOR-006 e interfaces), el horario se exporta **en el formato de trabajo del Encargado Docente, completando día, bloque y sala de cada sección, sin pérdida ni alteración de registros, y solo si no tiene choques**. Banner no tiene interfaz ni admite carga masiva: hoy el horario se digita registro por registro. Por eso el entregable son **tablas** (no XML —solo es el formato interno de FET— ni PDF).

Cada versión **sin choques** produce `Horario_Vxx.xlsx`, que es **una copia del libro original del cliente** (se conservan fórmulas, valores, textos `N/A` y las hojas `academicos`, `laboratorios` y `ayudantes`, así que el archivo se puede volver a importar) en la que solo cambia lo autorizado:

| Hoja | Qué cambia / contiene |
|---|---|
| `asignaturas` | Solo se completa la columna `Horario` (notación del cliente: `Ma F (V3-Rob), Lu DE (U8)`); se agregan al final `Horario_original` y `Estado_programacion` |
| `guia_digitacion` | **Guía para digitar en Banner** (no es una carga automática): una fila por sección y sesión, con periodo, identificador estable, NRC, día, bloque, horas, sala, docentes (con porcentajes) y estado de preparación |
| `resumen` | Parámetros, semillas, indicadores, supuestos y pendientes de confirmar |
| `choques` | Choques detectados (vacía si la versión es aprobable) |

y `guia_digitacion_Vxx.csv` (UTF-8 con BOM). Las versiones con choques **no se exportan**: solo se deja un `Choques_Vxx.csv` de diagnóstico.

In [ ]:
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

def rango_hora(h, dur): return f"{HORAS[BLOQUES[h]][0]}–{HORAS[BLOQUES[h + dur - 1]][1]}"

def horario_texto(g):
    """Notación del cliente: 'Ma F (V3-Rob), Lu DE (U8)'."""
    return ", ".join(f"{DIA_CORTO[DIAS[r.Dia]]} {''.join(BLOQUES[r.Hora:r.Hora + r.Dur])}" + (f" ({r.Sala})" if r.Sala else "")
                     for r in g.sort_values(["Dia", "Hora"]).itertuples())

SUPUESTOS = [
    "Factible bajo supuestos: NO equivale a 'validado para operación' hasta confirmar los puntos siguientes con el Encargado Docente.",
    "Disponibilidad de profesores hora: se infiere por los DÍAS en que hoy ya dictan (no hay disponibilidad por bloque en la planilla).",
    "Capacidad de salas sin dato del cliente: ESTIMADA con el mayor cupo que el cliente ya ubicó allí (ver hoja de salas del notebook).",
    "Ocupación externa de salas (ciclo básico, otros departamentos) y prioridad de ciencias básicas: no vienen en la planilla, no modeladas.",
    "Pertenencia de estudiantes: los paralelos (C2, T#, L##) se modelan como subgrupos del semestre; confirmar cohortes con el cliente.",
    "Banner: sin formato de importación acreditado; esta hoja es una guía de digitación. Campos como campus, método de instrucción o fechas no están acreditados.",
]

def tablas_version(v, entry=None, etq=None, rank=None, extra_res=()):
    entry = entry or VERSIONES[v]; etq = etq or f"V{v:02d}"
    rank = int(resumen.loc[f"V{v:02d}", "Ranking"]) if rank is None else rank
    t = entry["t"]; por_clase = {cid: g for cid, g in t.groupby("ClaseId")}
    est = secciones["Motivo"].replace("", "PROGRAMADA").values
    filas = []
    for _, c in clases.iterrows():
        g = por_clase[c.ClaseId].sort_values(["Dia", "Hora"])
        for i in c.Filas:
            s = secciones.loc[i]; fila_xl = i + 2
            code_ = s["Code"] if pd.notna(s["Code"]) else "SINCODIGO"
            estado = "Pendiente: crear NRC en Banner" if str(s["NRC"]).upper() == "ADD" else ("Pendiente: falta código" if pd.isna(s["Code"]) else "Lista")
            for k, r in enumerate(g.itertuples(), 1):
                filas.append({"Periodo": PERIODO, "ID sección": f"{PERIODO}|{code_}-{s['P']}|fila{fila_xl}", "ID sesión": f"{PERIODO}|{code_}-{s['P']}|fila{fila_xl}|S{k}",
                              "NRC": s["NRC"], "Código": s["Code"], "Materia": s["Materia"], "Sección": s["P"], "Asignatura": s["Asignatura"],
                              "Programa": s["Programa"], "Semestre": s["Semestre"], "Tipo": s["Tipo"], "Liga": s["Liga"], "Cupo": s["Cupo"],
                              "Día": DIAS[r.Dia], "Bloque": "".join(BLOQUES[r.Hora:r.Hora + r.Dur]),
                              "Hora inicio": HORAS[BLOQUES[r.Hora]][0], "Hora término": HORAS[BLOQUES[r.Hora + r.Dur - 1]][1],
                              "Sala": r.Sala or "", "Docente(s) y porcentajes (planilla)": s["Profesor"],
                              "RUT(s) (planilla)": s["RUT"], "Estado de preparación": estado, "Fila en planilla": fila_xl})
    guia = pd.DataFrame(filas)
    guia["_d"] = guia["Día"].map(DIA_IDX)
    guia = guia.sort_values(["Programa", "Semestre", "Asignatura", "Sección", "_d", "Hora inicio"], na_position="last").drop(columns="_d")
    ind = entry["ind"]
    res = pd.DataFrame([("Versión", etq if entry is not VERSIONES.get(v) else f"{etq} de {N_VERSIONES}"), ("Periodo", PERIODO), ("Generado", dt.datetime.now().strftime("%d-%m-%Y %H:%M")),
                        ("Motor", "FET 7.10.5 (fet-cl)"), ("Semillas (6 componentes)", ", ".join(map(str, entry["corrida"]["semillas"]))),
                        ("MODO_SALAS", MODO_SALAS), ("DIAS_HORA_ACOTADOS", DIAS_HORA_ACOTADOS), ("FIJAR_SABADO", FIJAR_SABADO),
                        ("MAX_HUECOS_SEMANA", MAX_HUECOS_SEMANA), ("MAX_VENTANAS_DOCENTE", MAX_VENTANAS_DOCENTE),
                        ("TIEMPO_LIMITE_S", TIEMPO_LIMITE_S), ("Archivo de entrada", os.path.basename(XLSX))]
                       + list(ind.items()) + [("Ranking", rank)] + list(extra_res)
                       + [("Supuesto / pendiente", z) for z in SUPUESTOS], columns=["Parámetro / indicador", "Valor"])
    choq = pd.DataFrame({"Choque": entry["viol"] or ["(ninguno: versión aprobable bajo los supuestos declarados)"]})
    return est, guia, res, choq

def estilo_hoja(ws, df, congelar=True):
    if congelar: ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF"); cell.fill = PatternFill("solid", fgColor="0B3C6F"); cell.alignment = Alignment(wrap_text=True, vertical="center")
    for j, col in enumerate(df.columns, 1):
        ancho = max([len(str(col))] + [len(str(z)) for z in df[col].head(300)])
        ws.column_dimensions[get_column_letter(j)].width = min(max(9, ancho + 2), 60)

def df_a_hoja(wb, nombre, df):
    ws = wb.create_sheet(nombre)
    ws.append(list(df.columns))
    for fila in df.itertuples(index=False): ws.append([None if pd.isna(z) else (z.item() if hasattr(z, 'item') else z) for z in fila])   # pd.NA / NaN -> celda vacía
    estilo_hoja(ws, df)

COL_HORARIO = 25   # columna 'Horario' de la hoja 'asignaturas'

def exportar_xlsx_version(v, ruta, entry=None, etq=None, rank=None, extra_res=(), hojas_extra=None):
    """Copia del libro ORIGINAL (fórmulas y hojas intactas) con solo lo autorizado modificado."""
    entry = entry or VERSIONES[v]
    est, guia, res, choq = tablas_version(v, entry, etq, rank, extra_res)
    wb2 = openpyxl.load_workbook(XLSX)                 # data_only=False: conserva fórmulas
    ws = wb2["asignaturas"]
    assert ws.cell(1, COL_HORARIO).value == "Horario"
    n_col = ws.max_column
    ws.cell(1, n_col + 1, "Horario_original"); ws.cell(1, n_col + 2, "Estado_programacion")
    for r_ in range(2, ws.max_row + 1):
        ws.cell(r_, n_col + 1, ws.cell(r_, COL_HORARIO).value); ws.cell(r_, n_col + 2, est[r_ - 2])
    por_clase = {cid: g for cid, g in entry["t"].groupby("ClaseId")}
    for _, c in clases.iterrows():
        h = horario_texto(por_clase[c.ClaseId])
        for i in c.Filas:
            assert str(ws.cell(i + 2, 1).value) == str(raw.at[i, "NRC"]), f"fila desalineada: {i + 2}"
            ws.cell(i + 2, COL_HORARIO, h)
    for cell in ws[1][n_col:n_col + 2]: cell.font = Font(bold=True)
    df_a_hoja(wb2, "guia_digitacion", guia); df_a_hoja(wb2, "resumen", res); df_a_hoja(wb2, "choques", choq)
    for nombre_h, df_h in (hojas_extra or {}).items(): df_a_hoja(wb2, nombre_h, df_h)
    wb2.save(ruta)
    return guia

def verificar_integridad(ruta):
    """Compara celda a celda el archivo GUARDADO contra el original (fórmulas incluidas)."""
    a, b = openpyxl.load_workbook(XLSX), openpyxl.load_workbook(ruta)
    assert a.sheetnames == b.sheetnames[:len(a.sheetnames)], "cambió el orden o los nombres de las hojas originales"
    difs = 0
    for nombre in a.sheetnames:
        wa, wb_ = a[nombre], b[nombre]
        for fila in wa.iter_rows():
            for cel in fila:
                if nombre == "asignaturas" and cel.column == COL_HORARIO and cel.row > 1: continue   # única columna autorizada
                if cel.value != wb_[cel.coordinate].value: difs += 1
    assert difs == 0, f"{difs} celdas originales fueron alteradas"
    # reimportación: el libro exportado debe poder leerse otra vez con el mismo ETL
    re_sec = limpiar_secciones(pd.read_excel(ruta, sheet_name="asignaturas", dtype=object), academ)
    assert (re_sec["Motivo"] == "").sum() == (secciones["Motivo"] == "").sum(), "la reimportación cambia las secciones válidas"
    return True

SALIDA = os.path.join(PROJECT_DIR, "salida_horarios", dt.datetime.now().strftime("%Y%m%d_%H%M%S"))
os.makedirs(SALIDA, exist_ok=True)
APROBABLES = [v for v in sorted(VERSIONES) if not VERSIONES[v]["viol"]]
for v in sorted(VERSIONES):
    if v not in APROBABLES:
        pd.DataFrame({"Choque": VERSIONES[v]["viol"]}).to_csv(os.path.join(SALIDA, f"Choques_V{v:02d}.csv"), index=False, encoding="utf-8-sig")
        print(f"  V{v:02d}: {len(VERSIONES[v]['viol'])} choques -> NO se exporta (solo Choques_V{v:02d}.csv de diagnóstico)"); continue
    ruta = os.path.join(SALIDA, f"Horario_V{v:02d}.xlsx")
    guia = exportar_xlsx_version(v, ruta)
    verificar_integridad(ruta)                                      # sobre el archivo ya guardado
    guia.to_csv(os.path.join(SALIDA, f"guia_digitacion_V{v:02d}.csv"), index=False, encoding="utf-8-sig")
    print(f"  Horario_V{v:02d}.xlsx: {len(guia)} filas en 'guia_digitacion'; integridad vs. original verificada celda a celda; reimportable ✔")
assert APROBABLES, "Ninguna versión quedó sin choques: no hay nada que exportar (ver Choques_Vxx.csv)"
resumen.to_csv(os.path.join(SALIDA, "00_Resumen_Versiones.csv"), encoding="utf-8-sig")
print("Carpeta de salida:", SALIDA)
display(guia.head(12))

## 6b. Replanificación parcial: cambiar una asignatura y generar nuevas propuestas (HOR-005)

Caso de uso: *«la asignatura X debe cambiar de horario; ¿qué alternativas tenemos sin desarmar todo lo demás?»*. Se toma una **versión base** (por defecto la mejor del ranking), se declara el **cambio** y FET regenera solo la parte afectada:

1. Las clases **no afectadas** quedan **fijadas** exactamente donde estaban (día, bloque y sala) — el ERS pide que las asignaciones fijadas no cambien.
2. La clase modificada recibe la restricción pedida (no usar su horario actual, evitar ciertos días, solo ciertos bloques, forzar una sala…).
3. Con `LIBERAR = "relacionadas"` también se liberan las clases que comparten docente o semestre con la modificada, para que el cambio tenga espacio de solución (si se deja en `"solo"`, únicamente se mueve la clase modificada).
4. Se generan `N_ALTERNATIVAS` propuestas con semillas distintas, cada una verificada con los mismos **choques** de siempre, y se informa **qué secciones cambiaron respecto de la base**. Se exportan como `Replan_Rxx.xlsx` (mismo formato que las versiones, más una hoja `cambios_vs_base`).

In [ ]:
# ------------------------------ QUÉ CAMBIAR (editable) ------------------------------
BASE_VERSION   = None        # None = la mejor versión del ranking; o un número (1, 2, …)
LIBERAR        = "relacionadas"   # "solo": se mueve únicamente la clase modificada | "relacionadas": + clases con el mismo docente o semestre
N_ALTERNATIVAS = 4
CAMBIOS = [
    dict(nrc="21949",                    # NRC de la sección (columna NRC de la planilla) — Proyecto Integrador Software
         evitar_horario_actual="bloque", # None | "bloque": no puede quedar en el mismo día y bloque | "dia": no puede quedar en el mismo día
         prohibir_dias=[],               # p. ej. ["Viernes"]
         solo_dias=None,                 # p. ej. ["Martes", "Jueves"]
         solo_bloques=None,              # p. ej. ["A", "B", "C"] (bloques de inicio permitidos)
         sala=None),                     # p. ej. ["Y1-004"]: fuerza la sala
]

In [ ]:
def ejecutar_replanificacion():
    aprob = [v for v in VERSIONES if not VERSIONES[v]["viol"]]
    assert aprob, "No hay versión base sin choques"
    base = BASE_VERSION or int(resumen.drop(index="Cliente (ref.)", errors="ignore").loc[[f"V{v:02d}" for v in aprob], "Ranking"].astype(float).idxmin()[1:])
    tb = VERSIONES[base]["t"].set_index("Id")
    print(f"Versión base: V{base:02d}")

    # ---- clases modificadas y relacionadas
    def clases_de(nrc): return [c for _, c in clases.iterrows() if str(nrc) in c.NRC.split(" / ")]
    mod = {}
    for cb in CAMBIOS:
        cs = clases_de(cb["nrc"]); assert cs, f"NRC {cb['nrc']} no está entre las clases programadas"
        for c in cs: mod[c.ClaseId] = (c, cb)
    libres = set(mod)
    if LIBERAR == "relacionadas":
        ruts = {r for c, _ in mod.values() for r, _ in c.Docentes}
        yrs = {y for c, _ in mod.values() for y in (s.split("-S")[0] + "-S" + s.split("-S")[1].split("-")[0] for s in sets_estudiantes(c))}
        for _, c in clases.iterrows():
            if {r for r, _ in c.Docentes} & ruts or {s.split("-S")[0] + "-S" + s.split("-S")[1].split("-")[0] for s in sets_estudiantes(c)} & yrs:
                libres.add(c.ClaseId)
    print(f"Clases modificadas: {[clase_por_id[i].Asignatura + ' (NRC ' + clase_por_id[i].NRC + ')' for i in mod]}")
    print(f"Clases liberadas para reacomodarse: {len(libres)} de {len(clases)}; fijadas exactamente como en la base: {len(clases) - len(libres)}")

    # ---- restricciones
    fijos = {a.Id: (int(tb.loc[a.Id, "Dia"]), int(tb.loc[a.Id, "Hora"]), tb.loc[a.Id, "Sala"]) for a in acts.itertuples() if a.ClaseId not in libres}
    permitidos, forzadas = {}, {}
    for cid, (c, cb) in mod.items():
        for a in acts[acts.ClaseId == cid].itertuples():
            d0, h0 = int(tb.loc[a.Id, "Dia"]), int(tb.loc[a.Id, "Hora"])
            ok = {(d, h) for d in range(5) for h in inicios_validos(a.Dur)}
            if cb.get("solo_dias"): ok = {(d, h) for d, h in ok if DIAS[d] in cb["solo_dias"]}
            if cb.get("prohibir_dias"): ok = {(d, h) for d, h in ok if DIAS[d] not in cb["prohibir_dias"]}
            if cb.get("solo_bloques"): ok = {(d, h) for d, h in ok if BLOQUES[h] in cb["solo_bloques"]}
            if cb.get("evitar_horario_actual") == "bloque": ok.discard((d0, h0))
            if cb.get("evitar_horario_actual") == "dia": ok = {(d, h) for d, h in ok if d != d0}
            permitidos[a.Id] = ok
            if cb.get("sala"): forzadas[a.Id] = list(cb["sala"])
    fet_r = os.path.join(RUN_DIR, "replanificacion.fet")
    escribir_fet(fet_r, fijos=fijos, permitidos=permitidos, salas_forzadas=forzadas)

    with ThreadPoolExecutor(max_workers=min(N_ALTERNATIVAS, os.cpu_count() or 2)) as ex:
        corr = list(ex.map(lambda i: ejecutar_version(i, fet_input=fet_r, tag="r", offset=500), range(1, N_ALTERNATIVAS + 1)))
    alts = {}
    for c in corr:
        print(f"  R{c['version']:02d}: {'OK ' if c['ok'] else 'SIN SOLUCIÓN'} {c['seg']:.1f}s")
        if not c["ok"]: continue
        t = tabla_horario(cargar_solucion(c["xml"])); viol, ind = verificar(t)
        # cumplimiento de lo pedido y de lo fijado
        for a in acts.itertuples():
            r = t[t.Id == a.Id].iloc[0]
            if a.Id in fijos and (r.Dia, r.Hora, r.Sala) != fijos[a.Id]: viol.append(f"Se movió una asignación fijada: {r.Asignatura} ({r.NRC})")
            if a.Id in permitidos and (r.Dia, r.Hora) not in permitidos[a.Id]: viol.append(f"No se respetó el cambio pedido en {r.Asignatura} ({r.NRC})")
        cambios = []
        for r in t.itertuples():
            b = tb.loc[r.Id]
            if (r.Dia, r.Hora, r.Sala) != (b.Dia, b.Hora, b.Sala):
                cambios.append({"NRC": r.NRC, "Asignatura": r.Asignatura, "Sección": r.Seccion, "Docente(s)": r.Docentes, "Sesión": r.Sesion,
                                "Antes": f"{DIA_CORTO[DIAS[int(b.Dia)]]} {''.join(BLOQUES[int(b.Hora):int(b.Hora) + r.Dur])} {b.Sala}".strip(),
                                "Ahora": f"{DIA_CORTO[DIAS[r.Dia]]} {''.join(BLOQUES[r.Hora:r.Hora + r.Dur])} {r.Sala}".strip(),
                                "Motivo": "cambio pedido" if r.ClaseId in mod else "reacomodo por el cambio"})
        cdf = pd.DataFrame(cambios, columns=["NRC", "Asignatura", "Sección", "Docente(s)", "Sesión", "Antes", "Ahora", "Motivo"])
        ind = dict(ind, **{"Sesiones cambiadas vs base": len(cdf), "Clases cambiadas vs base": cdf[["NRC", "Sección"]].drop_duplicates().shape[0]})
        alts[c["version"]] = dict(t=t, viol=list(dict.fromkeys(viol)), ind=ind, corrida=c, cambios=cdf)
    return base, alts, mod

BASE, ALTERNATIVAS, MODIFICADAS = ejecutar_replanificacion()
assert ALTERNATIVAS, "FET no encontró alternativas: relaja el cambio (p. ej. LIBERAR='relacionadas', menos restricciones o más TIEMPO_LIMITE_S)"
res_alt = pd.DataFrame({f"R{k:02d}": d["ind"] for k, d in ALTERNATIVAS.items()}).T
res_alt["Puntaje (menor = mejor)"] = res_alt["Choques"] * 1000 + res_alt["Sesiones cambiadas vs base"] * 2 + res_alt["Bloques libres estudiantes"] + 0.5 * res_alt["Ventanas docentes"]
res_alt["Ranking"] = res_alt["Puntaje (menor = mejor)"].rank(method="min").astype(int)
print("\nAlternativas (se prefiere: sin choques, pocos cambios respecto de la base, buena compacidad):")
display(res_alt.sort_values("Ranking"))
print("Alternativas distintas entre sí:", len({tuple(sorted((r.Id, r.Dia, r.Hora, r.Sala) for r in d["t"].itertuples())) for d in ALTERNATIVAS.values()}), "de", len(ALTERNATIVAS))
mejor_alt = int(res_alt["Ranking"].astype(float).idxmin()[1:])
print(f"\nCambios de la mejor alternativa (R{mejor_alt:02d}) respecto de la base V{BASE:02d}:")
display(ALTERNATIVAS[mejor_alt]["cambios"])

In [ ]:
SALIDA_R = SALIDA          # misma carpeta de salida que las versiones
for k, d in sorted(ALTERNATIVAS.items()):
    etq = f"R{k:02d}"
    if d["viol"]:
        pd.DataFrame({"Choque": d["viol"]}).to_csv(os.path.join(SALIDA_R, f"Choques_{etq}.csv"), index=False, encoding="utf-8-sig")
        print(f"  {etq}: {len(d['viol'])} choques -> NO se exporta (Choques_{etq}.csv)"); continue
    ruta = os.path.join(SALIDA_R, f"Replan_{etq}.xlsx")
    extra = [("Versión base", f"V{BASE:02d}"), ("Cambio pedido", json.dumps(CAMBIOS, ensure_ascii=False)), ("LIBERAR", LIBERAR)]
    guia = exportar_xlsx_version(k, ruta, entry=d, etq=etq, rank=int(res_alt.loc[etq, "Ranking"]), extra_res=extra, hojas_extra={"cambios_vs_base": d["cambios"]})
    verificar_integridad(ruta)
    guia.to_csv(os.path.join(SALIDA_R, f"guia_digitacion_{etq}.csv"), index=False, encoding="utf-8-sig")
    print(f"  Replan_{etq}.xlsx: {len(d['cambios'])} sesiones cambiadas vs V{BASE:02d}; integridad vs. original verificada; reimportable ✔")
res_alt.to_csv(os.path.join(SALIDA_R, "00_Resumen_Replanificacion.csv"), encoding="utf-8-sig")

## 6c. (Opcional) PDFs ordenados por versión — `GENERAR_PDF = True`

Para revisar a ojo (no para Banner). Cada versión genera un PDF con **portada e índice con números de página** (y marcadores navegables en el lector de PDF) y cuatro partes en orden fijo:

1. **Horario por programa y semestre** (ICCI, IenCI, ICEL, PC, …; dentro de cada programa, por semestre).
2. **Horario por docente** (orden alfabético por apellido).
3. **Ocupación por sala** (orden alfabético).
4. **Listado de secciones** (por programa, semestre y asignatura, con NRC, docente, sala y bloques).

Además se genera `00_Resumen_Versiones.pdf` (comparativo).

In [ ]:
from reportlab.lib.pagesizes import A4, landscape
from reportlab.lib import colors
from reportlab.lib.units import mm
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.platypus import (BaseDocTemplate, PageTemplate, Frame, Paragraph, Spacer, Table, TableStyle, PageBreak,
                                NextPageTemplate)
from reportlab.platypus.tableofcontents import TableOfContents
from reportlab.platypus.flowables import KeepTogether

UCN_AZUL, UCN_GRIS = colors.HexColor("#0B3C6F"), colors.HexColor("#5A6B7B")
_ss = getSampleStyleSheet()
ST = dict(
    titulo=ParagraphStyle("titulo", parent=_ss["Title"], fontSize=24, leading=28, textColor=UCN_AZUL, alignment=0),
    sub=ParagraphStyle("sub", parent=_ss["Normal"], fontSize=11, leading=14, textColor=UCN_GRIS),
    h1=ParagraphStyle("h1", parent=_ss["Heading1"], fontSize=15, textColor=UCN_AZUL, spaceBefore=0, spaceAfter=2),
    h2=ParagraphStyle("h2", parent=_ss["Heading2"], fontSize=13, textColor=UCN_AZUL, spaceBefore=0, spaceAfter=1),
    sec=ParagraphStyle("sec", parent=_ss["Heading1"], fontSize=15, textColor=UCN_AZUL, spaceAfter=4),
    nota=ParagraphStyle("nota", parent=_ss["Normal"], fontSize=8.5, leading=10.5, textColor=UCN_GRIS),
    celda=ParagraphStyle("celda", parent=_ss["Normal"], fontSize=7, leading=8.2, alignment=1),
    txt=ParagraphStyle("txt", parent=_ss["Normal"], fontSize=9.5, leading=12.5),
    tabla=ParagraphStyle("tabla", parent=_ss["Normal"], fontSize=7.4, leading=8.8),
)

def color_clase(cid):
    r = random.Random(cid * 7919)
    h = r.random(); import colorsys
    rr, gg, bb = colorsys.hls_to_rgb(h, 0.86, 0.55)
    return colors.Color(rr, gg, bb)

def apellido_orden(nombre):
    partes = nombre.split(); return " ".join(partes[1:] + partes[:1]).lower()

def texto_sesiones(g):
    return "; ".join(f"{DIA_CORTO[DIAS[r.Dia]]} {''.join(BLOQUES[r.Hora:r.Hora + r.Dur])}" for r in g.sort_values(["Dia", "Hora"]).itertuples())

def grilla(t_sub, ancho, con_docente=True, con_sala=True, con_prog=False):
    """Tabla semanal (bloques x días) con las sesiones de t_sub; las sesiones dobles se combinan en una celda."""
    usa_sab = (t_sub.Dia == 5).any()
    nd = 6 if usa_sab else 5
    celdas = {}
    for r in t_sub.itertuples():
        for k in range(r.Dur): celdas.setdefault((r.Dia, r.Hora + k), []).append(r)
    data = [[""] + [DIAS[d] for d in range(nd)]]
    for h in range(N_H):
        data.append([Paragraph(f"<b>{BLOQUES[h]}</b><br/>{HORAS[BLOQUES[h]][0]}–{HORAS[BLOQUES[h]][1]}", ST["celda"])] + [""] * nd)
    estilo = [("GRID", (0, 0), (-1, -1), .4, colors.HexColor("#B8C2CC")), ("BACKGROUND", (0, 0), (-1, 0), UCN_AZUL),
              ("TEXTCOLOR", (0, 0), (-1, 0), colors.white), ("FONTSIZE", (0, 0), (-1, 0), 8.5), ("ALIGN", (0, 0), (-1, 0), "CENTER"),
              ("BACKGROUND", (0, 1), (0, -1), colors.HexColor("#EEF2F6")), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
              ("LINEBELOW", (0, 3), (-1, 3), 1.4, UCN_GRIS)]      # separa mañana / tarde (almuerzo)
    for (d, h), l in celdas.items():
        if d >= nd: continue
        partes = []
        for r in l:
            lin = f"<b>{x(r.Asignatura[:44])}</b>" + (f" · {r.Seccion}" if r.Seccion != "C1" else "")
            if con_prog and r.Programas: lin += f"<br/>{x(r.Programas)}"
            if con_docente: lin += f"<br/>{x(r.Docentes[:46])}"
            if con_sala and r.Sala: lin += f"<br/>Sala {r.Sala}"
            partes.append(lin)
        data[h + 1][d + 1] = Paragraph("<br/>".join(partes), ST["celda"])
        estilo.append(("BACKGROUND", (d + 1, h + 1), (d + 1, h + 1), color_clase(l[0].ClaseId) if len(l) == 1 else colors.HexColor("#F4C7C3")))
    # combinar sesiones dobles
    for (d, h), l in celdas.items():
        if d < nd and len(l) == 1 and l[0].Dur == 2 and l[0].Hora == h:
            estilo.append(("SPAN", (d + 1, h + 1), (d + 1, h + 2)))
            data[h + 2][d + 1] = ""
    wcol = [24 * mm] + [(ancho - 24 * mm) / nd] * nd
    tb = Table(data, colWidths=wcol, rowHeights=[8 * mm] + [17.5 * mm] * N_H, repeatRows=1)
    tb.setStyle(TableStyle(estilo)); return tb

class Doc(BaseDocTemplate):
    def afterFlowable(self, fl):
        if isinstance(fl, Paragraph) and fl.style.name in ("h1", "h2"):
            nivel = 0 if fl.style.name == "h1" else 1
            texto = fl.getPlainText(); key = f"k{self.seq.nextf('bm')}"
            self.canv.bookmarkPage(key); self.canv.addOutlineEntry(texto, key, nivel, closed=(nivel == 1))
            self.notify("TOCEntry", (nivel, texto, self.page, key))

def pie(canv, doc, etiqueta):
    canv.saveState(); canv.setFont("Helvetica", 7.5); canv.setFillColor(UCN_GRIS)
    canv.drawString(15 * mm, 8 * mm, f"UCN · DISC — Oferta 2026-20 — {etiqueta}")
    canv.drawRightString(landscape(A4)[0] - 15 * mm, 8 * mm, f"Página {doc.page}"); canv.restoreState()

In [ ]:
def pdf_version(v, ruta_pdf, resumen_v):
    t = VERSIONES[v]["t"]; etiqueta = f"Versión {v:02d}"
    W, Hh = landscape(A4); ancho = W - 30 * mm
    doc = Doc(ruta_pdf, pagesize=landscape(A4), leftMargin=15 * mm, rightMargin=15 * mm, topMargin=12 * mm, bottomMargin=13 * mm,
              title=f"Horario DISC 2026-20 — {etiqueta}", author="Equipo Capstone — plataforma de apoyo a la programación académica")
    doc.addPageTemplates([PageTemplate(id="p", frames=[Frame(15 * mm, 13 * mm, ancho, Hh - 25 * mm, id="f", leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)],
                                       onPage=lambda c, d: pie(c, d, etiqueta))])
    S = []
    # ---- portada
    S += [Spacer(1, 6 * mm), Paragraph("Propuesta de horario — Oferta académica 2026-20", ST["titulo"]),
          Paragraph(f"Departamento de Ingeniería de Sistemas y Computación · UCN · <b>{etiqueta}</b> de {N_VERSIONES}", ST["sub"]), Spacer(1, 4 * mm)]
    ind = VERSIONES[v]["ind"]; c = VERSIONES[v]["corrida"]
    filas = [["Generado", dt.datetime.now().strftime("%d-%m-%Y %H:%M")], ["Motor", "FET 7.10.5 (fet-cl), semillas " + ", ".join(map(str, c["semillas"]))],
             ["Clases / sesiones semanales", f"{len(clases)} clases · {len(t)} sesiones"],
             ["Choques (verificador independiente)", f"{ind['Choques']}  " + ("✔ aprobable bajo supuestos" if ind["Choques"] == 0 else "✘ NO aprobable")],
             ["Bloques libres de estudiantes (menor = mejor)", str(ind["Bloques libres estudiantes"])],
             ["Ventanas de docentes (menor = mejor)", str(ind["Ventanas docentes"])],
             ["Días promedio por docente", str(ind["Días promedio por docente"])],
             ["Sesiones en bloques F–G (tarde)", str(ind["Sesiones en bloques F-G"])],
             ["Coincidencia con el horario original del cliente", f"{ind['% igual al horario del cliente']} %"],
             ["Ranking entre versiones", f"{int(resumen_v['Ranking'])} de {N_VERSIONES}"]]
    tb = Table([[Paragraph(f"<b>{a}</b>", ST["txt"]), Paragraph(b, ST["txt"])] for a, b in filas], colWidths=[95 * mm, ancho - 95 * mm])
    tb.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), .4, colors.HexColor("#B8C2CC")), ("BACKGROUND", (0, 0), (0, -1), colors.HexColor("#EEF2F6")),
                            ("VALIGN", (0, 0), (-1, -1), "MIDDLE")]))
    S += [tb, Spacer(1, 5 * mm)]
    toc = TableOfContents(); toc.levelStyles = [ParagraphStyle("t0", fontSize=10, leading=13, leftIndent=0, fontName="Helvetica-Bold"),
                                                 ParagraphStyle("t1", fontSize=8.5, leading=10.5, leftIndent=12)]
    S += [Paragraph("Contenido", ST["sec"]), toc, PageBreak()]

    def pagina(titulo, subtitulo, tabla, nivel="h2"):
        return [Paragraph(titulo, ST[nivel]), Paragraph(subtitulo, ST["nota"]), Spacer(1, 2 * mm), tabla, PageBreak()]

    # ---- parte 1: por programa / semestre
    S += [Paragraph("Parte 1 · Horario por programa y semestre", ST["h1"]), Paragraph("Asignaturas obligatorias agrupadas por programa y semestre; las secciones paralelas (C2, T#, laboratorios L##) aparecen en el mismo cuadro con su sección. Celdas rojas: superposición entre secciones paralelas.", ST["nota"]), PageBreak()]
    years = sorted({s.rsplit("-", 1)[0] if s.count("-") == 2 else s for st_ in t.Sets for s in st_}, key=lambda y: (y.split("-")[0], int(y.split("-S")[1])))
    for y in years:
        sub = t[t.Sets.map(lambda L: any(s == y or s.startswith(y + "-") for s in L))]
        S += pagina(f"{y.split('-')[0]} · Semestre {y.split('-S')[1]}", f"{sub.ClaseId.nunique()} clases · {len(sub)} sesiones semanales", grilla(sub, ancho, con_prog=False))
    libres = t[t.Sets.map(len) == 0]
    S += [Paragraph("Electivos, posgrado y clases sin semestre", ST["h2"]), Paragraph("Clases sin grupo de estudiantes obligatorio (electivos y programas de posgrado): solo se protege docente y sala.", ST["nota"]), PageBreak()]
    for prog, sub in sorted(libres.groupby("Programas"), key=lambda kv: kv[0]):
        S += pagina(f"Electivos · {prog or 's/i'}", f"{sub.ClaseId.nunique()} clases", grilla(sub, ancho, con_prog=True))

    # ---- parte 2: docentes
    S += [Paragraph("Parte 2 · Horario por docente", ST["h1"]), Paragraph("Orden alfabético por apellido. Bloques con dos clases del mismo docente indicarían un choque (no ocurre en versiones aprobables).", ST["nota"]), PageBreak()]
    for rut in sorted(DOCENTES, key=lambda r: apellido_orden(DOCENTES[r]["nombre"])):
        sub = t[t.RUTs.map(lambda L: rut in L)]
        if sub.empty: continue
        d = DOCENTES[rut]
        S += pagina(d["nombre"], f"RUT {rut} · contrato: {d['contrato']} · {sub.ClaseId.nunique()} clases · {int(sub.Dur.sum())} bloques semanales", grilla(sub, ancho, con_docente=False, con_prog=True))

    # ---- parte 3: salas
    S += [Paragraph("Parte 3 · Ocupación por sala", ST["h1"]), Paragraph("Orden alfabético. Las capacidades sin dato del cliente son estimadas.", ST["nota"]), PageBreak()]
    for sala in sorted(t.Sala[t.Sala != ""].unique()):
        sub = t[t.Sala == sala]
        S += pagina(f"Sala {sala}", f"Capacidad: {CAP_EFECTIVA[sala]} · {int(sub.Dur.sum())} de {N_H * 5} bloques semanales ocupados ({100 * sub.Dur.sum() / (N_H * 5):.0f} %)", grilla(sub, ancho, con_sala=False, con_prog=True))

    # ---- parte 4: listado
    S += [Paragraph("Parte 4 · Listado de secciones", ST["h1"]), Spacer(1, 2 * mm)]
    orden = t.assign(_p=t.Programas.str.split(" / ").str[0], _s=t.Sets.map(lambda L: int(L[0].split("-S")[1].split("-")[0]) if L else 99)).sort_values(["_p", "_s", "Asignatura", "Seccion", "Dia", "Hora"])
    filas = [[Paragraph(f"<b>{h}</b>", ST["tabla"]) for h in ["Programa", "Sem", "NRC", "Asignatura", "Sec.", "Docente(s)", "Sala", "Horario"]]]
    for (cid), g in orden.groupby("ClaseId", sort=False):
        r = g.iloc[0]
        filas.append([Paragraph(x(r.Programas), ST["tabla"]), Paragraph(str(r._s) if r._s != 99 else "–", ST["tabla"]), Paragraph(x(r.NRC), ST["tabla"]),
                      Paragraph(x(r.Asignatura), ST["tabla"]), Paragraph(r.Seccion, ST["tabla"]), Paragraph(x(r.Docentes), ST["tabla"]),
                      Paragraph(", ".join(sorted({s for s in g.Sala if s})) or "–", ST["tabla"]),
                      Paragraph("; ".join(f"{DIA_CORTO[DIAS[q.Dia]]} {rango_hora(q.Hora, q.Dur)}" for q in g.sort_values(['Dia', 'Hora']).itertuples()), ST["tabla"])])
    tl = Table(filas, colWidths=[22 * mm, 11 * mm, 22 * mm, 66 * mm, 10 * mm, 58 * mm, 20 * mm, ancho - 209 * mm], repeatRows=1)
    tl.setStyle(TableStyle([("GRID", (0, 0), (-1, -1), .3, colors.HexColor("#B8C2CC")), ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#DCE5EE")),
                            ("VALIGN", (0, 0), (-1, -1), "TOP"), ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7F9FB")])]))
    S.append(tl)
    doc.multiBuild(S)

def pdf_resumen(ruta_pdf):
    W, Hh = landscape(A4); ancho = W - 30 * mm
    doc = Doc(ruta_pdf, pagesize=landscape(A4), leftMargin=15 * mm, rightMargin=15 * mm, topMargin=12 * mm, bottomMargin=13 * mm, title="Resumen de versiones de horario")
    doc.addPageTemplates([PageTemplate(id="p", frames=[Frame(15 * mm, 13 * mm, ancho, Hh - 25 * mm, id="f", leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0)],
                                       onPage=lambda c, d: pie(c, d, "Resumen comparativo"))])
    r = resumen.copy(); r = r.sort_values("Ranking", na_position="last")
    cab = ["Versión", "Choques", "Bloques libres\nestudiantes", "Ventanas\ndocentes", "Días prom.\npor docente", "Sesiones\nF–G", "% igual\nal cliente", "Ranking"]
    filas = [cab] + [[i, int(row["Choques"]), int(row["Bloques libres estudiantes"]), int(row["Ventanas docentes"]), row["Días promedio por docente"],
                      int(row["Sesiones en bloques F-G"]), row["% igual al horario del cliente"], "–" if pd.isna(row["Ranking"]) else int(row["Ranking"])] for i, row in r.iterrows()]
    tb = Table(filas, colWidths=[38 * mm] + [(ancho - 38 * mm) / 7] * 7)
    tb.setStyle(TableStyle([("BACKGROUND", (0, 0), (-1, 0), UCN_AZUL), ("TEXTCOLOR", (0, 0), (-1, 0), colors.white), ("GRID", (0, 0), (-1, -1), .4, colors.HexColor("#B8C2CC")),
                            ("ALIGN", (1, 0), (-1, -1), "CENTER"), ("FONTSIZE", (0, 0), (-1, -1), 9), ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
                            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7F9FB")])]))
    nota = ("<b>Cómo leer esta tabla.</b> Todas las versiones aprobables tienen 0 choques según un verificador independiente del motor. "
            "El ranking ordena por bloques libres de estudiantes + 0,5 × ventanas de docentes (menor es mejor). «% igual al cliente» mide cuántas sesiones coinciden con el "
            "día y bloque de la planilla original: un valor bajo es normal, pues FET reconstruye el horario desde cero. La fila «Cliente (ref.)» es la planilla original pasada por el mismo verificador: "
            "los choques que reporta son <i>posibles</i> conflictos a revisar (algunos podrían ser clases conjuntas sin liga declarada).")
    S = [Paragraph("Comparativo de versiones de horario — Oferta 2026-20", ST["h1"]), Spacer(1, 3 * mm), tb, Spacer(1, 5 * mm), Paragraph(nota, ST["txt"])]
    if len(sol_cliente) == len(acts) and v_cli:
        S += [Spacer(1, 4 * mm), Paragraph("Posibles conflictos detectados en la planilla original", ST["h2"])]
        S += [Paragraph("• " + x(m), ST["txt"]) for m in v_cli]
    doc.build(S)

In [ ]:
if GENERAR_PDF:
    for v in sorted(VERSIONES):
        ruta = os.path.join(SALIDA, f"Horario_V{v:02d}.pdf")
        pdf_version(v, ruta, resumen.loc[f"V{v:02d}"]); print(f"  Horario_V{v:02d}.pdf  ({os.path.getsize(ruta)/1024:.0f} KB)")
    pdf_resumen(os.path.join(SALIDA, "00_Resumen_Versiones.pdf"))
else:
    print("GENERAR_PDF=False: se omiten los PDF (los entregables son las tablas Excel/CSV).")

## 7. Descargar

In [ ]:
ZIP = os.path.join(os.path.dirname(SALIDA), f"Horarios_FET_{os.path.basename(SALIDA)}.zip")
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(os.path.join(SALIDA, "*"))): z.write(f, os.path.join("Horarios_FET", os.path.basename(f)))
    z.write(FET_INPUT, "Horarios_FET/oferta_2026-20.fet")          # el problema completo, reabrible en la GUI de FET
print("ZIP con todas las tablas (y PDF si se generaron) y el archivo .fet:", ZIP, f"({os.path.getsize(ZIP)/1e6:.1f} MB)")

if IN_COLAB:
    from google.colab import files
    files.download(ZIP)
else:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(ZIP), result_html_prefix="Descargar todo (.zip): "))
    for f in sorted(glob.glob(os.path.join(SALIDA, "*.pdf"))): display(FileLink(os.path.relpath(f), result_html_prefix="PDF: "))

## 8. Alcance, supuestos y siguientes pasos

- **Qué demuestra**: con los datos reales del cliente, `fet-cl` genera en segundos horarios completos y **sin choques** (verificados de forma independiente), y permite obtener alternativas cambiando solo la semilla.
- **Supuestos que hay que confirmar con el Encargado Docente** (todos parametrizables arriba o en el ETL):
  1. La *disponibilidad* de profesores hora se aproxima con los días en que hoy ya dictan (la planilla no trae disponibilidad explícita).
  2. La capacidad declarada es siempre el límite. Las salas sin dato llevan una capacidad **estimada** con el mayor cupo que el cliente ya ubicó allí. `Y1-005` figura con 20 pero el cliente ya ubica secciones de 21: se reporta como inconsistencia y el modelo no usa esa sala para ellas.
  3. Las salas `R-0xx`, `Y1-001`, `Y1-104`, `Y2-107`, `U8` pueden ser compartidas con otras unidades (ciclo básico, otros departamentos): la planilla no informa sus horas ocupadas, por lo que no están modeladas.
  4. El modo `MODO_SALAS="cliente"` respeta las salas que el cliente ya asignó a cada asignatura; `"libre"` permite cualquier sala con capacidad suficiente.
  5. Preferencias de docentes de planta y la restricción de *ciencias básicas* (prioridad de sus horarios) no vienen en la planilla; son insumo pendiente del ERS (HOR-001).
- **Secciones excluidas** (sección 2.4): inactivas, «No se dictará» y tesis/talleres sin horario ni bloques. Deben confirmarse con el cliente.
  6. Pertenencia de estudiantes en paralelos (`C2`, `T#`, `L##`): se modelan como subgrupos del semestre; falta confirmar las cohortes.
- **«Aprobable» = factible bajo estos supuestos**, no «validado para operación». Solo se exportan versiones sin choques, y el Excel exportado conserva el libro original (fórmulas y hojas), de modo que puede reimportarse.
- **Pendiente (requiere al cliente):** formato real de ingreso a Banner (la hoja `guia_digitacion` es una guía manual), replanificación parcial (HOR-005: fijar asignaciones y comparar contra una versión base) y disponibilidad por bloque de profesores hora.
- **Módulo predictivo, presupuesto y Banner** quedan fuera de este notebook (ver anteproyecto v3 y ERS).